In [1]:
import pandas as pd
import json
import os

# ==========================================
# 1. LOAD FINAL ASPECT DICTIONARY
# ==========================================
print("📚 Loading aspect dictionary...")
dict_path = "../data/usefuldict/final_aspects.json" # Adjust path if necessary
with open(dict_path, 'r', encoding='utf-8') as f:
    aspect_dict = json.load(f)

print(f"✅ Loaded {len(aspect_dict)} aspects")

# ==========================================
# 2. LOAD MULTI-GENERATIONAL DATA
# ==========================================
DATA_DIR = "../data/processed/"
files = [
    "a52samsungcleanedytcomment.csv",
    "a53samsungcleanedytcomment.csv",
    "a54samsungcleanedytcomment.csv",
    "a55samsungcleanedytcomment.csv",
    "a56samsungcleanedytcomment.csv"
]

all_dfs = []
print("\n📝 Loading comments from all generations...")
for file_name in files:
    file_path = os.path.join(DATA_DIR, file_name)
    if os.path.exists(file_path):
        df_temp = pd.read_csv(file_path)
        
        # 🌟 CRITICAL FIX: TAG THE GENERATION (A52, A53, etc.) 🌟
        df_temp['generation'] = file_name[:3].upper() 
        all_dfs.append(df_temp)

# Combine all datasets into one big DataFrame
df = pd.concat(all_dfs, ignore_index=True)

# Detect the correct comment column
comment_col = 'clean_comment' if 'clean_comment' in df.columns else df.select_dtypes(include=['object']).columns[0]
print(f"📋 Columns available: {list(df.columns)}")
print(f"📝 Using column: '{comment_col}'")
print(f"📊 Total combined comments: {len(df)}")

# ==========================================
# 3. ASPECT EXTRACTION PROCESS
# ==========================================
print("\n🔍 Processing comments for Aspect Extraction...")

results = []
processed_count = 0

for index, row in df.iterrows():
    comment = str(row[comment_col]).lower()
    generation = row['generation'] # Grab the generation tag
    
    detected_aspects = set()
    matched_keywords = set()
    
    # Check against the dictionary
    for aspect, keywords in aspect_dict.items():
        for keyword in keywords:
            # If the keyword is mentioned in the comment
            if keyword.lower() in comment:
                detected_aspects.add(aspect)
                matched_keywords.add(keyword)
                
    # If we found at least one aspect, save the row
    if detected_aspects:
        results.append({
            'comment_id': index,
            'generation': generation,                   # <--- NOW INCLUDED IN RESULTS
            'original_comment': comment,
            'detected_aspects': ", ".join(list(detected_aspects)),
            'aspect_count': len(detected_aspects),
            'matched_keywords': ", ".join(list(matched_keywords))
        })
        
    processed_count += 1
    if processed_count % 1000 == 0:
        print(f"  Processed {processed_count}/{len(df)} comments...")

# ==========================================
# 4. SAVE RESULTS FOR INDOBERT CLASSIFICATION
# ==========================================
hasil_df = pd.DataFrame(results)
print(f"\n✅ Done! Found {len(hasil_df)} comments with aspects")
print(f"📊 Coverage: {(len(hasil_df)/len(df))*100:.1f}% of comments have aspects")

# Ensure directory exists
os.makedirs("../data/comment_and_aspects", exist_ok=True)

# Save to CSV
output_csv = "../data/comment_and_aspects/aspect_detection_results.csv"
hasil_df.to_csv(output_csv, index=False)
print(f"\n📁 CSV saved to: {output_csv}")
print(f"📊 CSV shape: {hasil_df.shape}")

print("\n🔍 Sample results (Notice the new 'generation' column):")
display(hasil_df.head())

📚 Loading aspect dictionary...
✅ Loaded 14 aspects

📝 Loading comments from all generations...
📋 Columns available: ['clean_comment', 'generation']
📝 Using column: 'clean_comment'
📊 Total combined comments: 62007

🔍 Processing comments for Aspect Extraction...
  Processed 1000/62007 comments...
  Processed 2000/62007 comments...
  Processed 3000/62007 comments...
  Processed 4000/62007 comments...
  Processed 5000/62007 comments...
  Processed 6000/62007 comments...
  Processed 7000/62007 comments...
  Processed 8000/62007 comments...
  Processed 9000/62007 comments...
  Processed 10000/62007 comments...
  Processed 11000/62007 comments...
  Processed 12000/62007 comments...
  Processed 13000/62007 comments...
  Processed 14000/62007 comments...
  Processed 15000/62007 comments...
  Processed 16000/62007 comments...
  Processed 17000/62007 comments...
  Processed 18000/62007 comments...
  Processed 19000/62007 comments...
  Processed 20000/62007 comments...
  Processed 21000/62007 comm

,comment_id,generation,original_comment,detected_aspects,aspect_count,matched_keywords
0,0,A52,why galaxy a with helio g processor is awesome...,Performance,1,"snapdragon, snap, lag, gaming, performa"
1,1,A52,gw dapet di harga rb hasil tt sama infinix hot...,"Features, Price, Storage, Connectivity",4,"nfc, harga, ram"
2,4,A52,secen di harga kemahalan ngak ya,Price,1,"harga, mahal, kemahalan"
3,7,A52,dpt harga brp bg,Price,1,harga
4,10,A52,alhamdulillah bisa nonton video ini pakai gala...,Camera,1,video


In [2]:
hasil_df.head()


,comment_id,original_comment,detected_aspects,aspect_count,matched_keywords
0,1,btw om saat review performance di game berat i...,"Camera, Performance, Design, BuildQuality",4,"video, berat, drop"
1,3,dan saya jadi orang terakhir yg nonton video i...,Camera,1,video
2,6,userhplama ini baru aja dilem ulang kaget juga...,Connectivity,1,kaget
3,9,body polikarbonat mate tetep bisa jamuran ga sih,"Security, BuildQuality",2,polikarbonat
4,14,habis beli samsung a dan apakah layarnya meman...,"Design, Display",2,"layar, warna"
